<a href="https://colab.research.google.com/github/lanaozeki08-jpg/SERS_GS2/blob/main/SERS_GS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install openai

import json
from getpass import getpass
from openai import OpenAI

from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

MODELO = "gpt-4o-mini"

def calcular_consumo_energia(
    moradores=4,
    minutos_banho=15,
    minutos_banho_economico=10,
    potencia_chuveiro=5.5,
    potencia_economica=3.5,
    tarifa_kwh=0.85,
    dias_mes=30,
    cobertura_solar=0.90,
    investimento_solar=5000
):
    """Calcula consumo, economia, emissões e retorno do solar."""

    consumo_atual = (
        moradores * minutos_banho / 60
        * potencia_chuveiro * dias_mes
    )

    consumo_habitos = (
        moradores * minutos_banho_economico / 60
        * potencia_economica * dias_mes
    )

    consumo_solar = consumo_atual * (1 - cobertura_solar)
    consumo_combinado = consumo_habitos * (1 - cobertura_solar)

    cenarios = {
        "Atual": consumo_atual,
        "Mudança de hábitos": consumo_habitos,
        "Aquecimento solar": consumo_solar,
        "Hábitos + solar": consumo_combinado
    }

    resultados = {}

    for nome, consumo in cenarios.items():
        resultados[nome] = {
            "consumo_kwh_mes": round(consumo, 2),
            "custo_mensal": round(consumo * tarifa_kwh, 2),
            "economia_mensal": round(
                (consumo_atual - consumo) * tarifa_kwh, 2
            ),
            "reducao_percentual": round(
                (consumo_atual - consumo) / consumo_atual * 100, 1
            ),
            "emissoes_kg_co2e": round(consumo * 0.0648, 3)
        }

    economia_solar = (
        (consumo_atual - consumo_solar) * tarifa_kwh
    )

    economia_solar_com_habitos = (
        (consumo_habitos - consumo_combinado) * tarifa_kwh
    )

    payback_solar = (
        round(investimento_solar / economia_solar, 1)
        if economia_solar > 0 else None
    )

    payback_combinado = (
        round(investimento_solar / economia_solar_com_habitos, 1)
        if economia_solar_com_habitos > 0 else None
    )

    return {
        "premissas": {
            "moradores": moradores,
            "minutos_banho": minutos_banho,
            "potencia_chuveiro_kw": potencia_chuveiro,
            "tarifa_kwh": tarifa_kwh,
            "dias_mes": dias_mes,
            "cobertura_solar": cobertura_solar,
            "investimento_solar": investimento_solar
        },
        "resultados": resultados,
        "payback_meses": {
            "solar_isolado": payback_solar,
            "solar_depois_dos_habitos": payback_combinado
        }
    }

ferramentas = [
    {
        "type": "function",
        "function": {
            "name": "calcular_consumo_energia",
            "description": (
                "Calcula o consumo e o custo de energia de uma residência, "
                "compara chuveiro elétrico e aquecimento solar, estima "
                "economia, emissões e payback. Use sempre que o usuário "
                "pedir simulações, cálculos ou comparações numéricas."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "moradores": {
                        "type": "integer",
                        "description": "Número de moradores da residência."
                    },
                    "minutos_banho": {
                        "type": "number",
                        "description": "Minutos de banho por pessoa por dia."
                    },
                    "minutos_banho_economico": {
                        "type": "number",
                        "description": "Tempo de banho econômico em minutos."
                    },
                    "potencia_chuveiro": {
                        "type": "number",
                        "description": "Potência atual do chuveiro em kW."
                    },
                    "potencia_economica": {
                        "type": "number",
                        "description": "Potência alternativa do chuveiro em kW."
                    },
                    "tarifa_kwh": {
                        "type": "number",
                        "description": "Tarifa de energia em reais por kWh."
                    },
                    "dias_mes": {
                        "type": "integer",
                        "description": "Quantidade de dias considerados."
                    },
                    "cobertura_solar": {
                        "type": "number",
                        "description": "Cobertura solar entre 0 e 1."
                    },
                    "investimento_solar": {
                        "type": "number",
                        "description": "Investimento estimado em reais."
                    }
                },
                "required": [],
                "additionalProperties": False
            }
        }
    }
]


instrucoes = """
Você é o Citizen Assist AI, um assistente virtual especializado
em eficiência energética e sustentabilidade residencial.

Seu objetivo é ajudar famílias a entender seu consumo de energia,
comparar alternativas e tomar decisões mais econômicas e sustentáveis.

Você deve:

1. Conversar de maneira natural, educada e acessível.
2. Fazer perguntas quando faltarem informações importantes.
3. Explicar o que significa kWh, tarifa, consumo e economia.
4. Comparar chuveiros elétricos com aquecimento solar térmico.
5. Orientar sobre redução do tempo de banho e uso consciente de energia.
6. Explicar vantagens, desvantagens e limitações de cada alternativa.
7. Utilizar a ferramenta de cálculo sempre que o usuário pedir
   simulações, valores, comparações numéricas ou payback.
8. Nunca inventar resultados numéricos.
9. Explicar que a cobertura solar de 90% é uma hipótese de simulação,
   não uma garantia de desempenho.
10. Informar que os valores de investimento são estimativas hipotéticas,
    e que o custo real depende de orçamento e dimensionamento técnico.

Premissas iniciais do projeto:
- Família de 4 pessoas.
- 15 minutos de banho por pessoa.
- Chuveiro de 5,5 kW.
- Tarifa estimada de R$ 0,85/kWh.
- 30 dias por mês.
- Alternativa econômica: 10 minutos de banho e chuveiro de 3,5 kW.
- Cobertura solar hipotética de 90%.
- Investimento de referência de R$ 5.000.

Aquecimento solar térmico aquece água diretamente. Não confunda
esse sistema com painéis fotovoltaicos que geram eletricidade.

Quando o usuário não informar dados, você pode começar usando
as premissas iniciais, deixando claro que são estimativas.
"""

historico = [
    {
        "role": "system",
        "content": instrucoes
    }
]


def responder(mensagem):

    historico.append({
        "role": "user",
        "content": mensagem
    })

    while True:

        resposta = client.chat.completions.create(
            model=MODELO,
            messages=historico,
            tools=ferramentas,
            tool_choice="auto",
            temperature=0.4
        )

        mensagem_ia = resposta.choices[0].message

        # Se a IA não solicitar ferramentas, retorna a resposta
        if not mensagem_ia.tool_calls:

            historico.append({
                "role": "assistant",
                "content": mensagem_ia.content or ""
            })

            return mensagem_ia.content or ""

        historico.append(
            mensagem_ia.model_dump(exclude_none=True)
        )

        for chamada in mensagem_ia.tool_calls:

            try:
                if chamada.function.name == "calcular_consumo_energia":

                    argumentos = json.loads(
                        chamada.function.arguments
                    )

                    resultado = calcular_consumo_energia(
                        **argumentos
                    )

                else:
                    resultado = {
                        "erro": "Ferramenta não encontrada."
                    }

            except Exception as erro:
                resultado = {
                    "erro": str(erro)
                }

            historico.append({
                "role": "tool",
                "tool_call_id": chamada.id,
                "content": json.dumps(
                    resultado,
                    ensure_ascii=True
                )
            })



print("CITIZEN ASSIST AI - ASSISTENTE DE ENERGIA")

print("\nOlá! Sou o Citizen Assist AI!")
print("Posso ajudar você a economizar energia e entender")
print("alternativas mais sustentáveis para sua residência.")
print("\nVocê pode perguntar, por exemplo:")
print("- Quanto minha família gasta com chuveiro elétrico?")
print("- Quanto economizaria com aquecimento solar?")
print("- Em quanto tempo recuperaria o investimento?")
print("- Como posso diminuir minha conta de energia?")
print("\nDigite 'sair' para encerrar a conversa.")
print("-" * 60)

while True:

    pergunta = input("\nVocê: ").strip()

    if pergunta.lower() in ["sair", "encerrar", "tchau"]:
        print("\nCitizen Assist AI: Obrigado pela conversa!")
        print("Até a próxima! Vamos construir um futuro mais sustentável.")
        break

    if not pergunta:
        continue

    try:
        resposta = responder(pergunta)
        print("\nCitizen Assist AI:", resposta)

    except Exception as erro:
        print("\nNão consegui processar sua mensagem.")
        print("Verifique sua chave da API e tente novamente.")
        print("Detalhe do erro:", erro)


CITIZEN ASSIST AI - ASSISTENTE DE ENERGIA

Olá! Sou o Citizen Assist AI!
Posso ajudar você a economizar energia e entender
alternativas mais sustentáveis para sua residência.

Você pode perguntar, por exemplo:
- Quanto minha família gasta com chuveiro elétrico?
- Quanto economizaria com aquecimento solar?
- Em quanto tempo recuperaria o investimento?
- Como posso diminuir minha conta de energia?

Digite 'sair' para encerrar a conversa.
------------------------------------------------------------

Você: 	Quanto minha família de 3 pessoas gasta com banho?

Citizen Assist AI: Para calcular o gasto com banho, preciso de algumas informações:

1. Quantos minutos cada pessoa passa no banho por dia?
2. Qual é a potência do chuveiro que você utiliza em kW?
3. Qual é a tarifa de energia que você paga por kWh?

Se você não tiver esses dados, posso usar algumas estimativas. O que você prefere?

Você: 15 minutos, 5,5kW, tarifa estimada

Citizen Assist AI: Aqui estão os resultados do seu gasto com b